# Relatório Final - Data Science
## Bacharelado em Ciência da Computação / PUCPR (2026-1)

**Prof. Rayson Laroca**

Alander Menezes Arantes de Ávila - menezes.alander@pucpr.edu.br

Giancarlo Nunes Perli - giancarlo.perli@sanrocco.com.br

Gustavo Faria Cardoso - faria.cardoso@pucpr.edu.br

Paulo Henrique Perin - paulo.perin@pucpr.edu.br

Pedro Lucas Ghezzi Bittencourt - pedro.bittencourt@pucpr.edu.br

# Importações

In [23]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import skew as _skew, kurtosis as _kurt
from matplotlib.patches import Ellipse as MplEllipse, Patch
from matplotlib.lines import Line2D

#### O dataset foi adquirido através do Kaggle, é mantido pelo Laboratório de Propulsão a Jato do Instituto de Tecnologia da California, uma organização sob a NASA, e contém os seguintes atributos:

**Identificadores** — `id`, `spkid`, `full_name`, `pdes`, `name`, `prefix`, `orbit_id`: identificam entradas no catálogo, sem valor preditivo.

**Classificação**
- `neo` — *Near-Earth Object*: indica se o asteroide orbita na região próxima à Terra (Y/N).
- `pha` — *Potentially Hazardous Asteroid*: indica se o asteroide é classificado como potencialmente perigoso (Y/N). ⭐ **Variável-alvo do projeto.**

**Características físicas**
- `H` — Magnitude absoluta: medida de brilho intrínseco do asteroide. Quanto menor o valor, mais brilhante e geralmente maior o objeto.
- `diameter` — Diâmetro físico em km.
- `albedo` — Fração da luz solar refletida pela superfície (0 = completamente escuro, 1 = completamente reflexivo).

**Parâmetros orbitais**
- `e` — Excentricidade: descreve o formato da órbita. 0 = circular, próximo de 1 = muito alongada.
- `a` — Semi-eixo maior (AU): distância média do asteroide ao Sol.
- `q` — Periélio (AU): ponto mais próximo do Sol na órbita.
- `ad` — Afélio (AU): ponto mais distante do Sol na órbita.
- `i` — Inclinação (°): ângulo entre o plano da órbita do asteroide e o plano da eclíptica (plano da órbita da Terra).
- `w` — Argumento do periélio (°): orientação do ponto mais próximo ao Sol dentro do plano orbital.
- `n` — Movimento médio (°/dia): velocidade angular média ao longo da órbita.
- `per_y` — Período orbital em anos: tempo para completar uma volta ao redor do Sol.
- `moid` — Distância mínima de interseção orbital (AU): menor distância geométrica possível entre a órbita do asteroide e a da Terra.
- `moid_ld` — Mesmo que `moid`, expresso em distâncias lunares.

**Qualidade do ajuste orbital**
- `sigma_e` — Incerteza formal na excentricidade estimada.
- `rms` — Resíduo do ajuste orbital (arcsec): indica a qualidade do ajuste da órbita calculada às observações reais.

**Outros** — `class`: classe orbital dinâmica (MBA, APO, ATE, AMO, IEO…)

## **Objetivos do Projeto**

**Objetivo principal:** Construir um modelo de Machine Learning capaz de classificar, a partir de características físicas e/ou orbitais, se um asteroide é potencialmente perigoso (`pha = Y`), de forma que seja capaz de generalizar para asteroides não classificados.

**Objetivos secundários:**
- Identificar quais características físicas e orbitais melhor diferenciam PHAs dos demais asteroides.
- Avaliar a viabilidade das variáveis físicas (`diameter`, `albedo`, `H`) como features, considerando a alta taxa de valores ausentes.
- Investigar o desbalanceamento de classes e suas implicações para a modelagem.
- Comparar modelos com e sem `moid_ld`, avaliando se os parâmetros orbitais restantes são suficientes para a classificação.

## **Hipóteses**

- **H1:** Asteroides próximos a Terra tendem a ser mais perigosos.
- **H2:** Asteroides com maiores dimensões físicas são mais perigosos.
- **H3:** A geometria da órbita determina se um asteroide cruza a região da Terra.

# Carregamento dos Dados

In [24]:
ds = pd.read_csv(
    "https://github.com/aland3r/asteroides/releases/download/dataset/asteroids-dataset.csv",
    low_memory=False
)
print(f"Shape original: {ds.shape}")

Shape original: (958524, 45)


In [25]:
ds['neo'].value_counts()

neo
N    935625
Y     22895
Name: count, dtype: int64

# Limpeza e Tratamento

O dataset completo possui 958.524 registros e 45 colunas. O primeiro passo da limpeza foi remover as colunas identificadoras (`id`, `spkid`, `full_name`, `pdes`, `name`, `prefix` e `orbit_id`), que não carregam informação analítica. Em seguida, removemos os registros sem rótulo definido para asteroides potencialmente perigosos (`pha = NaN`), já que a variável alvo é indispensável para as análises supervisionadas.

Em seguida, removemos as colunas `diameter`, `albedo`, `diameter_sigma` e `diameter_bc`, todas com mais de 85% de valores ausentes. Antes de removê-las, verificamos se os valores presentes estavam concentrados nos PHAs, o que indicaria relevância preditiva. Não era o caso: a grande maioria dos valores pertencia à classe negativa (135.988 não-PHAs contra 221 PHAs para `diameter`), confirmando que a remoção não compromete a identificação de asteroides perigosos.

Por fim, filtramos o dataset para conter apenas NEOs (`neo = Y`). Como todo PHA é necessariamente um objeto próximo à Terra, os asteroides com `neo = N` nunca poderiam ser classificados como perigosos — sua inclusão apenas adicionaria ruído ao problema. Esse filtro reduz o dataset de 938.603 para 22.895 instâncias.

Durante a análise exploratória, identificamos registros com valores fisicamente implausíveis: 1 registro com `rms > 100` e 2 registros com `ad > 100 AU`. Esses valores indicam soluções orbitais mal determinadas e foram removidos iterativamente após sua identificação.

In [26]:
ds.isnull().sum()

id                     0
spkid                  0
full_name              0
pdes                   0
name              936460
prefix            958506
neo                    4
pha                19921
H                   6263
diameter          822315
albedo            823421
diameter_sigma    822443
orbit_id               0
epoch                  0
epoch_mjd              0
epoch_cal              0
equinox                0
e                      0
a                      0
q                      0
i                      0
om                     0
w                      0
ma                     1
ad                     4
n                      0
tp                     0
tp_cal                 0
per                    4
per_y                  1
moid               19921
moid_ld              127
sigma_e            19922
sigma_a            19922
sigma_q            19922
sigma_i            19922
sigma_om           19922
sigma_w            19922
sigma_ma           19922
sigma_ad           19926


In [62]:
df_clean = ds.dropna(subset=['pha'])

id_cols = ['id', 'spkid', 'full_name', 'pdes', 'name', 'prefix', 'orbit_id']
high_missing = ['diameter', 'albedo', 'diameter_sigma', 'diameter_bc']
df_clean = df_clean.drop(columns=id_cols + high_missing, errors='ignore')
df_clean = df_clean[df_clean['neo'] == 'Y']

# Remoção de valores fisicamente implausíveis identificados na análise exploratória
df_clean = df_clean[df_clean['rms'] <= 100]
df_clean = df_clean[df_clean['ad'] <= 100]

df_clean = df_clean.sample(frac=1, random_state=42).reset_index(drop=True)

print(df_clean.shape)
print(df_clean['pha'].value_counts())

(22891, 35)
pha
N    20825
Y     2066
Name: count, dtype: int64


### **Amostragem Balanceada**

O desbalanceamento de classes em `pha` dificulta a comparação visual entre os dois grupos. Para contornar isso, construímos `df_equal`: um subconjunto com igual número de PHAs e não-PHAs (2.066 de cada), totalizando 4.132 instâncias, com semente aleatória fixa para reprodutibilidade. Estatísticas populacionais e treinamento dos modelos utilizam `df_clean`; `df_equal` é reservado para visualizações exploratórias onde uma comparação visual justa entre as classes é necessária.

In [28]:
# df_clean contém apenas pha em {'Y','N'}

# separar classes
df_y = df_clean[df_clean['pha'] == 'Y'].copy()
df_n = df_clean[df_clean['pha'] == 'N'].copy()

# amostra balanceada: mesma quantidade de N que de Y
df_n_bal = df_n.sample(n=len(df_y), random_state=42)

# conjunto balanceado final
df_equal = pd.concat([df_y, df_n_bal], ignore_index=True)
df_equal = df_equal.sample(frac=1, random_state=42).reset_index(drop=True)

# verificação
print("df_equal shape:", df_equal.shape)
print(df_equal['pha'].value_counts())
print(df_equal['pha'].value_counts(normalize=True))

df_equal shape: (4132, 35)
pha
Y    2066
N    2066
Name: count, dtype: int64
pha
Y    0.5
N    0.5
Name: proportion, dtype: float64


---

# Descrição Estatística



Após a limpeza:

- 22.891 instâncias (2.066 PHAs e 20.825 não-PHAs);
- 35 atributos;
- **Problema:** classificação binária — `pha = Y` ou `pha = N`
- **Desbalanceamento:** apenas ~9% dos asteroides são PHAs entre os objetos próximos à Terra — qualquer modelo que preveja sempre "não perigoso" acertaria ~91% das vezes, tornando a acurácia uma métrica inadequada

In [ ]:
PHA_COLORS = {
    'Y': '#7A4A7E',  # Potentially Hazardous
    'N': '#2EAF8A'   # Non-Hazardous
}

---

# Análise de Dados Univariados

Esta seção analisa 20 variáveis do dataset de forma individual, combinando visualizações adequadas ao tipo de cada variável com estatísticas descritivas (média, mediana, desvio padrão, assimetria e curtose). Para variáveis contínuas, são identificadas distribuições, outliers, assimetrias e possíveis transformações. Ao final da seção, são definidas as variáveis de interesse para a modelagem.

#### **1. Qual é a proporção de asteroides potencialmente perigosos?**

Esperamos encontrar um desbalanceamento extremo — PHAs devem representar uma fração muito pequena do total, já que eventos de impacto são raros na natureza.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Proporção de Asteroides Potencialmente Perigosos (PHA)', fontsize=13, y=1.02)
datasets = [('Dataset Original', ds), ('Após Limpeza (NEOs)', df_clean)]
for ax, (titulo, data) in zip(axes, datasets):
    contagem = data['pha'].value_counts()
    sns.barplot(x=contagem.index, y=contagem.values, ax=ax, width=0.4,
                hue=contagem.index, palette=PHA_COLORS, legend=False)
    ax.set_title(titulo)
    ax.set_xlabel('PHA')
    ax.set_ylabel('Quantidade')
    ax.yaxis.grid(True, linestyle='--', alpha=0.7)
    ax.set_axisbelow(True)
    ax.spines['bottom'].set_visible(False)
    for i, (label, v) in enumerate(zip(contagem.index, contagem.values)):
        pct = v / len(data) * 100
        if label == 'N':
            ax.text(i, v / 2, f'{pct:.1f}%', ha='center', va='center', fontsize=12, fontweight='bold', color='white')
        else:
            ax.text(i, v + len(data) * 0.01, f'{v:,}\n({pct:.1f}%)', ha='center', va='bottom', fontsize=12, fontweight='bold', color='black')
plt.tight_layout()
plt.show()
print("Dataset original:")
print(ds['pha'].value_counts())
print(ds['pha'].value_counts(normalize=True).round(4))
print("\nApós limpeza:")
print(df_clean['pha'].value_counts())
print(df_clean['pha'].value_counts(normalize=True).round(4))

Há um desbalanceamento extremo no dataset original: apenas 0,2% dos asteroides (2.066) são potencialmente perigosos (`pha=Y`), contra 97,7% (936.537) não perigosos (`pha=N`). Após a limpeza e o filtro para objetos próximos à Terra (`neo=Y`), essa proporção vai para aproximadamente 9% de `pha=Y` e 91% de `pha=N`, ainda discrepante, porém mais tratável. Em ambos os casos, será necessário ajustar a distribuição de classes na etapa de pré-processamento. Métricas como acurácia isolada não serão suficientes para avaliar o modelo, sendo necessário considerar F1-score, precisão e recall.

**Problema de qualidade identificado:** desbalanceamento severo de classes, tratável com SMOTE ou undersampling na etapa de pré-processamento.

#### **2. Qual é a proporção de asteroides que passam perto da Terra?**

Esperamos que a grande maioria dos asteroides catalogados não sejam NEOs — o cinturão principal, localizado entre Marte e Júpiter, domina o sistema solar.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
contagem = ds['neo'].value_counts()
sns.barplot(x=contagem.index, y=contagem.values, ax=ax, width=0.4,
            hue=contagem.index, palette=PHA_COLORS, legend=False)
ax.set_title('Proporção de Objetos Próximos à Terra (NEO)', fontsize=13)
ax.set_xlabel('NEO')
ax.set_ylabel('Quantidade')
ax.spines['bottom'].set_visible(False)
ax.yaxis.grid(True, linestyle='--', alpha=0.7)
ax.set_axisbelow(True)
for i, (label, v) in enumerate(zip(contagem.index, contagem.values)):
    pct = v / len(ds) * 100
    if label == 'N':
        ax.text(i, v / 2, f'{pct:.1f}%', ha='center', va='center', fontsize=12, fontweight='bold', color='white')
    else:
        ax.text(i, v + 5000, f'{v:,}\n({pct:.1f}%)', ha='center', va='bottom', fontsize=12, fontweight='bold', color='black')
plt.tight_layout()
plt.show()

Apenas 2,4% dos asteroides (22.895) são objetos próximos à Terra (`neo=Y`), contra 97,6% (935.625) que orbitam longe (`neo=N`). Todo asteroide potencialmente perigoso (`pha=Y`) é necessariamente um NEO, mas nem todo NEO é um PHA. Essa relação motivou a decisão de filtrar o dataset para esse subconjunto na etapa de limpeza, reduzindo de 958.524 para 22.895 instâncias.

#### **3. Como se distribuem os asteroides entre as classes orbitais e quais concentram os mais perigosos?**

Esperamos que os asteroides do cinturão principal (MBA) dominem em número, mas que os PHAs estejam concentrados nas classes que cruzam a região próxima à Terra: APO, ATE, AMO e IEO.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Distribuição das Classes Orbitais', fontsize=13, y=1.02)
contagem = ds['class'].value_counts()
labels = list(contagem.index)
values = list(contagem.values)
COR_MBA = '#4A5F9E'
COR_OUTRAS = '#E89880'
posicoes = [0] + [0.5 + i * 0.8 for i in range(1, len(labels))]
for ax, log in zip(axes, [False, True]):
    for pos, label, v in zip(posicoes, labels, values):
        ax.bar(pos, v, width=1.2 if label == 'MBA' else 0.4, color=COR_MBA if label == 'MBA' else COR_OUTRAS)
    ax.set_xticks(posicoes); ax.set_xticklabels(labels)
    ax.set_xlim(-0.8, posicoes[-1] + 0.8)
    ax.set_xlabel('Classe Orbital'); ax.set_ylabel('Quantidade')
    ax.spines['bottom'].set_visible(False)
    ax.yaxis.grid(True, linestyle='--', alpha=0.7); ax.set_axisbelow(True)
    if log:
        ax.set_yscale('log'); ax.set_title('Escala Logarítmica')
    else:
        ax.set_title('Escala Original')
    for pos, label, v in zip(posicoes, labels, values):
        pct = v / len(ds) * 100
        if label == 'MBA':
            ax.text(pos, v/2 if not log else v*0.01, f'{pct:.1f}%', ha='center', va='center', fontsize=10, fontweight='bold', color='white')
        else:
            ax.text(pos, v*1.3 if log else v+5000, f'{pct:.1f}%', ha='center', va='bottom', fontsize=8, fontweight='bold', color='black')
from matplotlib.patches import Patch
for ax in axes:
    ax.legend(handles=[Patch(color=COR_MBA, label='MBA (dominante)'), Patch(color=COR_OUTRAS, label='Outras classes')], loc='upper right')
plt.tight_layout()
plt.show()

contagem_class_pha = df_clean.groupby(['class', 'pha']).size().unstack(fill_value=0)
contagem_class_pha = contagem_class_pha.sort_values('Y', ascending=False)
pct_pha = (contagem_class_pha['Y'] / contagem_class_pha.sum(axis=1) * 100).round(1)
print("% de PHAs por classe orbital (df_clean):"); print(pct_pha)

O dataset original é dominado por MBAs (89,3%). As classes APO, ATE, AMO e IEO são os únicos candidatos reais à classificação `pha=Y`. Entre os NEOs, os Apolos (`APO`) concentram o maior número absoluto de PHAs, mas Atens (`ATE`) e Atiras (`IEO`) têm maior proporção relativa — suas órbitas passam mais tempo dentro da região terrestre.

**Variável de interesse para modelagem:** `class` é uma feature categórica relevante, mas requer encoding antes da modelagem.

#### **4. Como se distribui o brilho intrínseco dos asteroides próximos à Terra (`H`) e ele diferencia os perigosos dos não perigosos?**

Esperamos que asteroides potencialmente perigosos sejam maiores e, portanto, mais brilhantes — ou seja, com valores menores de `H`.

In [ ]:
from scipy.stats import skew, kurtosis

data_H = df_clean['H'].dropna()
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle('Magnitude Absoluta (H) — Objetos Próximos à Terra', fontsize=13, y=1.02)
for label, grp in df_clean.groupby('pha'):
    d = grp['H'].dropna()
    sns.histplot(d, bins=25, kde=True, ax=axes[0], color=PHA_COLORS[label], edgecolor='#5A6A7A', alpha=0.5, label=f'pha={label}')
axes[0].axvline(22, color='#E07A5F', linestyle='--', linewidth=1.5, label='Limiar PHA (H=22)')
axes[0].set_title('Distribuição por Classe (pha=Y vs pha=N)')
axes[0].set_xlabel('H (magnitude)'); axes[0].set_ylabel('Contagem')
axes[0].legend(); axes[0].spines['bottom'].set_visible(False)
axes[0].yaxis.grid(True, linestyle='--', alpha=0.7); axes[0].set_axisbelow(True)
sns.boxplot(x=data_H, ax=axes[1], color='#4A5F9E', flierprops=dict(markerfacecolor='#8FA8D6', markersize=4))
axes[1].set_title('Tukey — Outliers'); axes[1].set_xlabel('H (magnitude)')
axes[1].spines['left'].set_visible(False); axes[1].xaxis.grid(True, linestyle='--', alpha=0.7); axes[1].set_axisbelow(True)
plt.tight_layout(); plt.show()
Q1=data_H.quantile(0.25); Q3=data_H.quantile(0.75); IQR=Q3-Q1
outliers=data_H[(data_H < Q1-1.5*IQR) | (data_H > Q3+1.5*IQR)]
print(f"Média: {data_H.mean():.2f} | Mediana: {data_H.median():.2f} | Desvio padrão: {data_H.std():.2f}")
print(f"Assimetria: {skew(data_H):.3f} | Curtose: {kurtosis(data_H):.3f}")
print(f"Outliers: {len(outliers)} ({len(outliers)/len(data_H)*100:.1f}%)")
print(data_H.describe())

A magnitude absoluta (`H`) mede o brilho intrínseco de um asteroide — quanto menor, maior o objeto. É um dos dois critérios formais de PHA (`H ≤ 22`). A distribuição nos NEOs é aproximadamente gaussiana, com média ~22,9 e desvio padrão ~3,0. O desvio padrão de ~3 magnitudes representa variação considerável de tamanho entre os NEOs.

PHAs concentram-se entre H=18–22 (no limite do critério), enquanto não-PHAs se deslocam para H=24–25. As duas classes ocupam faixas bem distintas. Apenas 13 outliers (0,1%) identificados pelo método de Tukey.

**Distribuição:** Aproximadamente gaussiana. **Assimetria:** levemente negativa nos PHAs. **Curtose:** próxima de 0 (mesocúrtica). **Transformação:** nenhuma necessária. **Variável de interesse — forte discriminador entre classes.**

#### **5. Qual fração dos asteroides próximos à Terra já passa orbitalmente perto o suficiente para ser perigosa (`moid`)?**

Esperamos que uma fração relevante dos NEOs já atenda ao critério de proximidade orbital, mas que nem todos sejam classificados como PHAs — o critério de tamanho deve ser o fator mais excludente.

In [ ]:
data_moid = df_clean['moid'].dropna()
p99 = data_moid.quantile(0.99)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle('Distância Mínima entre Órbitas (moid) — Objetos Próximos à Terra', fontsize=13, y=1.02)
sns.histplot(data_moid[data_moid <= p99], bins=40, kde=True, ax=axes[0], color='#4A5F9E', edgecolor='#5A6A7A')
axes[0].axvline(0.05, color='#E07A5F', linestyle='--', linewidth=1.5, label='Limiar PHA (0,05 AU)')
axes[0].set_title('Distribuição (até p99)'); axes[0].set_xlabel('MOID (AU)'); axes[0].set_ylabel('Contagem')
axes[0].legend(); axes[0].spines['bottom'].set_visible(False)
axes[0].yaxis.grid(True, linestyle='--', alpha=0.7); axes[0].set_axisbelow(True)
abaixo = (data_moid <= 0.05).sum(); acima = (data_moid > 0.05).sum()
wedges, texts, autotexts = axes[1].pie([abaixo, acima],
    labels=[f'MOID ≤ 0,05 AU\n({abaixo:,})', f'MOID > 0,05 AU\n({acima:,})'],
    autopct='%1.1f%%', colors=[PHA_COLORS['Y'], PHA_COLORS['N']], startangle=90)
for autotext in autotexts: autotext.set_color('white'); autotext.set_fontweight('bold')
axes[1].set_title('Proporção abaixo do limiar PHA')
plt.tight_layout(); plt.show()
print(f"Média: {data_moid.mean():.4f} | Mediana: {data_moid.median():.4f} | Desvio padrão: {data_moid.std():.4f}")
print(f"Assimetria: {skew(data_moid):.3f} | Curtose: {kurtosis(data_moid):.3f}")
print(data_moid.describe())

O `moid` representa a menor distância possível entre as órbitas da Terra e do asteroide. É um dos dois critérios formais de PHA: `moid ≤ 0,05 AU`. Notavelmente, 48,6% dos NEOs já atendem ao critério de proximidade, mas apenas ~9% são PHAs — o critério de tamanho (`H ≤ 22`) é o fator mais excludente. A distribuição é fortemente assimétrica à direita (assimetria alta).

**Distribuição:** Exponencial/assimétrica positiva. **Assimetria:** alta positiva. **Desvio padrão:** alto relativo à média, indicando grande variação de proximidade. **Possível transformação:** log para normalizar. **Variável de interesse — critério formal de PHA.**

#### **6. Quais são os valores extremos de distância mínima à Terra entre os objetos próximos à Terra (`moid_ld`)?**

Esperamos uma distribuição assimétrica à direita, com a maioria dos NEOs passando relativamente perto da Terra e uma minoria com órbitas mais afastadas.

In [ ]:
data_moid_ld = df_clean['moid_ld'].dropna()
p99 = data_moid_ld.quantile(0.99)
Q1=data_moid_ld.quantile(0.25); Q3=data_moid_ld.quantile(0.75); IQR=Q3-Q1
limite_superior = Q3 + 1.5*IQR
outliers = data_moid_ld[data_moid_ld > limite_superior]
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle('Distância Mínima à Órbita Terrestre (moid_ld)', fontsize=13, y=1.02)
sns.histplot(data_moid_ld[data_moid_ld <= p99], bins=40, kde=True, ax=axes[0], color='#4A5F9E', edgecolor='#5A6A7A')
axes[0].axvline(19.5, color='#E07A5F', linestyle='--', linewidth=1.5, label='Limiar PHA (19,5 LD)')
axes[0].set_title('Distribuição (até p99)'); axes[0].set_xlabel('Distâncias lunares'); axes[0].set_ylabel('Contagem')
axes[0].legend(); axes[0].spines['bottom'].set_visible(False)
axes[0].yaxis.grid(True, linestyle='--', alpha=0.7); axes[0].set_axisbelow(True)
sns.boxplot(x=data_moid_ld, ax=axes[1], color='#4A5F9E', flierprops=dict(markerfacecolor='#8FA8D6', markersize=4))
axes[1].set_title(f'Tukey — {len(outliers)} outliers ({len(outliers)/len(data_moid_ld)*100:.1f}%)')
axes[1].set_xlabel('Distâncias lunares'); axes[1].spines['left'].set_visible(False)
axes[1].xaxis.grid(True, linestyle='--', alpha=0.7); axes[1].set_axisbelow(True)
plt.tight_layout(); plt.show()
print(f"Média: {data_moid_ld.mean():.2f} | Mediana: {data_moid_ld.median():.2f} | Desvio padrão: {data_moid_ld.std():.2f}")
print(f"Assimetria: {skew(data_moid_ld):.3f} | Curtose: {kurtosis(data_moid_ld):.3f}")
print(f"Q1: {Q1:.2f} | Q3: {Q3:.2f} | Limite superior Tukey: {limite_superior:.2f}")

`moid_ld` é equivalente a `moid` em distâncias lunares (fator de conversão fixo: 1 AU ≈ 389,2 LD). O limiar PHA corresponde a 19,5 LD. A distribuição é fortemente assimétrica à direita, com 3,1% de outliers acima de 129,64 LD.

**Distribuição:** Exponencial/assimétrica positiva. **Nota:** `moid` e `moid_ld` são matematicamente equivalentes — na modelagem, manter apenas um para evitar multicolinearidade perfeita. **Possível transformação:** log. **Variável de interesse para modelagem.**

#### **7. Como se distribui o formato das órbitas dos asteroides (`e`) e isso diferencia os perigosos dos não perigosos?**

Esperamos que asteroides com órbitas mais alongadas (maior excentricidade) sejam mais propensos a cruzar a região terrestre e, portanto, mais frequentes entre os PHAs.

In [ ]:
data_e = df_clean.loc[(df_clean['e'] >= 0) & (df_clean['e'] < 1), 'e']
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('Excentricidade Orbital (e)', fontsize=13, y=1.02)
for ax, (titulo, data) in zip(axes[:2], [('Dataset Original', ds), ('Apenas NEOs', df_clean)]):
    d = data.loc[(data['e'] >= 0) & (data['e'] < 1), 'e']
    sns.histplot(d, bins=25, kde=True, ax=ax, color='#4A5F9E', edgecolor='#5A6A7A')
    ax.set_title(titulo); ax.set_xlabel('Excentricidade (e)'); ax.set_ylabel('Contagem')
    ax.spines['bottom'].set_visible(False); ax.yaxis.grid(True, linestyle='--', alpha=0.7); ax.set_axisbelow(True)
sns.violinplot(data=df_clean, x='pha', y='e', ax=axes[2], palette=PHA_COLORS)
axes[2].set_title('Distribuição por Classe (violin)'); axes[2].set_xlabel('PHA'); axes[2].set_ylabel('e')
axes[2].yaxis.grid(True, linestyle='--', alpha=0.7); axes[2].set_axisbelow(True)
plt.tight_layout(); plt.show()
Q1=data_e.quantile(0.25); Q3=data_e.quantile(0.75); IQR=Q3-Q1
outliers=data_e[data_e > Q3+1.5*IQR]
print(f"Média: {data_e.mean():.3f} | Mediana: {data_e.median():.3f} | Desvio padrão: {data_e.std():.3f}")
print(f"Assimetria: {skew(data_e):.3f} | Curtose: {kurtosis(data_e):.3f}")
print(f"Q1: {Q1:.2f} | Q3: {Q3:.2f} | Outliers acima de {Q3+1.5*IQR:.2f}: {len(outliers)} ({len(outliers)/len(data_e)*100:.1f}%)")

No dataset original, dominado por MBAs, a excentricidade concentra-se em 0,1–0,2. Nos NEOs, desloca-se para 0,4–0,6. O violin por classe revela separação clara: PHAs têm distribuição deslocada para valores mais altos.

**Distribuição:** Assimétrica positiva nos NEOs. **Assimetria:** moderada positiva. **Curtose:** próxima de 0. **Outliers:** 17 (0,1%) acima de 0,95 — fisicamente plausíveis. **Desvio padrão:** ~0,16, representando variação considerável de forma orbital. **Transformação:** nenhuma necessária. **Variável de interesse — forte discriminador entre classes.**

#### **8. Como se distribui o tamanho médio das órbitas dos asteroides próximos à Terra (`a`)?**

Esperamos uma distribuição bimodal, refletindo as subclasses de NEOs: Apolos e Atens com órbitas menores (~1,3 AU) e Amores com órbitas maiores (~2,1 AU).

In [ ]:
data_a = df_clean['a'].dropna()
p99 = data_a.quantile(0.99)
Q1=data_a.quantile(0.25); Q3=data_a.quantile(0.75); IQR=Q3-Q1
outliers=data_a[(data_a < Q1-1.5*IQR) | (data_a > Q3+1.5*IQR)]
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle('Semi-eixo Maior (a)', fontsize=13, y=1.02)
sns.histplot(data_a[data_a <= p99], bins=25, kde=True, ax=axes[0], color='#4A5F9E', edgecolor='#5A6A7A')
axes[0].set_title('Distribuição (até p99)'); axes[0].set_xlabel('Semi-eixo maior (AU)'); axes[0].set_ylabel('Contagem')
axes[0].spines['bottom'].set_visible(False); axes[0].yaxis.grid(True, linestyle='--', alpha=0.7); axes[0].set_axisbelow(True)
sns.boxplot(x=data_a[data_a <= p99], ax=axes[1], color='#4A5F9E', flierprops=dict(markerfacecolor='#8FA8D6', markersize=4))
axes[1].set_title(f'Tukey — {len(outliers)} outliers ({len(outliers)/len(data_a)*100:.1f}%)'); axes[1].set_xlabel('Semi-eixo maior (AU)')
axes[1].spines['left'].set_visible(False); axes[1].xaxis.grid(True, linestyle='--', alpha=0.7); axes[1].set_axisbelow(True)
plt.tight_layout(); plt.show()
print(f"Média: {data_a.mean():.3f} | Mediana: {data_a.median():.3f} | Desvio padrão: {data_a.std():.3f}")
print(f"Assimetria: {skew(data_a):.3f} | Curtose: {kurtosis(data_a):.3f}")
print(f"Q1: {Q1:.2f} | Q3: {Q3:.2f} | Outliers: {len(outliers)} ({len(outliers)/len(data_a)*100:.1f}%)")

A distribuição é bimodal, com picos em ~1,3 AU (APO/ATE) e ~2,1 AU (AMO). O desvio padrão (~0,7 AU) reflete essa variação entre subclasses. Apenas 48 outliers (0,2%) acima de 3,52 AU.

**Distribuição:** Bimodal, assimétrica à direita. **Assimetria:** moderada positiva. **Nota:** `a`, `ad`, `per_y` e `n` são matematicamente relacionados — multicolinearidade a tratar na modelagem. **Variável de interesse para modelagem.**

#### **9. Quão perto do Sol os asteroides próximos à Terra chegam em sua trajetória (`q`)?**

Esperamos que o periélio da maioria dos NEOs esteja próximo a 1 AU (órbita da Terra), com outliers abaixo de 0,36 AU representando asteroides com órbitas muito internas.

In [ ]:
data_q = df_clean['q'].dropna()
Q1=data_q.quantile(0.25); Q3=data_q.quantile(0.75); IQR=Q3-Q1
outliers=data_q[(data_q < Q1-1.5*IQR) | (data_q > Q3+1.5*IQR)]
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle('Distância do Periélio (q)', fontsize=13, y=1.02)
sns.histplot(data_q, bins=30, kde=True, ax=axes[0], color='#4A5F9E', edgecolor='#5A6A7A')
axes[0].axvline(1.0, color='#E07A5F', linestyle='--', linewidth=1.5, label='Órbita da Terra (1 AU)')
axes[0].set_title('Distribuição'); axes[0].set_xlabel('Periélio (AU)'); axes[0].set_ylabel('Contagem')
axes[0].legend(); axes[0].spines['bottom'].set_visible(False)
axes[0].yaxis.grid(True, linestyle='--', alpha=0.7); axes[0].set_axisbelow(True)
sns.boxplot(x=data_q, ax=axes[1], color='#4A5F9E', flierprops=dict(markerfacecolor='#8FA8D6', markersize=4))
axes[1].set_title(f'Tukey — {len(outliers)} outliers ({len(outliers)/len(data_q)*100:.1f}%)'); axes[1].set_xlabel('Periélio (AU)')
axes[1].spines['left'].set_visible(False); axes[1].xaxis.grid(True, linestyle='--', alpha=0.7); axes[1].set_axisbelow(True)
plt.tight_layout(); plt.show()
print(f"Média: {data_q.mean():.3f} | Mediana: {data_q.median():.3f} | Desvio padrão: {data_q.std():.3f}")
print(f"Assimetria: {skew(data_q):.3f} | Curtose: {kurtosis(data_q):.3f}")
print(f"Q1: {Q1:.2f} | Q3: {Q3:.2f} | Outliers: {len(outliers)} ({len(outliers)/len(data_q)*100:.1f}%)")

O IQR concentra-se entre 0,78–1,07 AU, próximo à órbita da Terra (1 AU). Os 548 outliers (2,4%) abaixo de 0,36 AU correspondem a Atens e Atiras com órbitas muito internas.

**Distribuição:** Assimétrica negativa (cauda à esquerda — objetos com periélio muito pequeno). **Assimetria:** negativa moderada. **Curtose:** moderada. **Variável parcialmente importante para modelagem.**

#### **10. Quão longe do Sol os asteroides próximos à Terra chegam em sua trajetória (`ad`)?**

Esperamos que o afélio da maioria dos NEOs se estenda entre 1,7 e 3,4 AU, com alguns outliers chegando além do cinturão principal.

In [ ]:
data_ad = df_clean['ad'].dropna()
Q1=data_ad.quantile(0.25); Q3=data_ad.quantile(0.75); IQR=Q3-Q1
outliers=data_ad[(data_ad < Q1-1.5*IQR) | (data_ad > Q3+1.5*IQR)]
fig, ax = plt.subplots(figsize=(9, 4))
sns.boxplot(x=data_ad, ax=ax, color='#4A5F9E', flierprops=dict(markerfacecolor='#8FA8D6', markersize=4))
ax.set_title(f'Distância do Afélio (ad) — {len(outliers)} outliers ({len(outliers)/len(data_ad)*100:.1f}%)')
ax.set_xlabel('Afélio (AU)'); ax.spines['left'].set_visible(False)
ax.xaxis.grid(True, linestyle='--', alpha=0.7); ax.set_axisbelow(True)
plt.tight_layout(); plt.show()
print(f"Média: {data_ad.mean():.3f} | Mediana: {data_ad.median():.3f} | Desvio padrão: {data_ad.std():.3f}")
print(f"Assimetria: {skew(data_ad):.3f} | Curtose: {kurtosis(data_ad):.3f}")
print(f"Q1: {Q1:.2f} | Q3: {Q3:.2f} | Limite superior Tukey: {Q3+1.5*IQR:.2f}")
print(f"Limite inferior: {Q1-1.5*IQR:.2f} (negativo — fisicamente impossível, sem outliers abaixo)")

Após remoção dos 2 registros acima de 100 AU, os 61 outliers remanescentes (0,3%) são fisicamente plausíveis. IQR entre 1,71–3,39 AU. O desvio padrão é alto devido à cauda positiva.

**Distribuição:** Assimétrica positiva. **Nota:** `ad = a × (1 + e)` — matematicamente derivado, portanto colinear com `a` e `e`. **Variável de baixa prioridade — provavelmente excluir da modelagem se `a` e `e` forem mantidos.**

#### **11. Como se distribui o ângulo das órbitas dos asteroides em relação ao plano da Terra (`i`) e isso diferencia os perigosos?**

Esperamos que PHAs tendam a órbitas com menor inclinação — quanto mais coplanar com a Terra, maior a chance de cruzamento orbital.

In [ ]:
data_i = df_clean['i']
Q1=data_i.quantile(0.25); Q3=data_i.quantile(0.75); IQR=Q3-Q1
outliers=data_i[data_i > Q3+1.5*IQR]
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle('Inclinação Orbital (i)', fontsize=13, y=1.02)
sns.histplot(data_i, bins=30, kde=True, ax=axes[0], color='#4A5F9E', edgecolor='#5A6A7A')
axes[0].set_title('Distribuição'); axes[0].set_xlabel('Inclinação (graus)'); axes[0].set_ylabel('Contagem')
axes[0].spines['bottom'].set_visible(False); axes[0].yaxis.grid(True, linestyle='--', alpha=0.7); axes[0].set_axisbelow(True)
sns.violinplot(data=df_clean, x='pha', y='i', ax=axes[1], palette=PHA_COLORS)
axes[1].set_title('Distribuição por Classe (violin)'); axes[1].set_xlabel('PHA'); axes[1].set_ylabel('Inclinação (graus)')
axes[1].yaxis.grid(True, linestyle='--', alpha=0.7); axes[1].set_axisbelow(True)
plt.tight_layout(); plt.show()
print(f"Média: {data_i.mean():.2f}° | Mediana: {data_i.median():.2f}° | Desvio padrão: {data_i.std():.2f}°")
print(f"Assimetria: {skew(data_i):.3f} | Curtose: {kurtosis(data_i):.3f}")
print(f"Outliers acima de {Q3+1.5*IQR:.2f}°: {len(outliers)} ({len(outliers)/len(data_i)*100:.1f}%)")

A distribuição é assimétrica à direita, com concentração entre 0°–30°. O violin por classe mostra que PHAs tendem a inclinações ligeiramente menores. Os 815 outliers (3,6%) têm órbitas muito inclinadas.

**Distribuição:** Assimétrica positiva (exponencial-like). **Assimetria:** positiva moderada-alta. **Curtose:** positiva (cauda pesada). **Desvio padrão:** ~11°. **Variável parcialmente importante para modelagem.**

#### **12. Como se distribui a orientação das órbitas dos asteroides em relação à eclíptica (`om`)?**

Esperamos uma distribuição aproximadamente uniforme entre 0° e 360° — não há direção preferencial no sistema solar para esse ângulo.

In [ ]:
data_om = df_clean['om'].dropna()
fig, ax = plt.subplots(figsize=(9, 4))
sns.histplot(data_om, bins=36, kde=True, ax=ax, color='#4A5F9E', edgecolor='#5A6A7A')
ax.set_title('Longitude do Nodo Ascendente (om)')
ax.set_xlabel('Longitude do nodo ascendente (graus)'); ax.set_ylabel('Contagem')
ax.spines['bottom'].set_visible(False); ax.yaxis.grid(True, linestyle='--', alpha=0.7); ax.set_axisbelow(True)
plt.tight_layout(); plt.show()
print(f"Média: {data_om.mean():.2f}° | Mediana: {data_om.median():.2f}° | Desvio padrão: {data_om.std():.2f}°")
print(f"Assimetria: {skew(data_om):.3f} | Curtose: {kurtosis(data_om):.3f}")

A distribuição é aproximadamente uniforme entre 0° e 360°, sem preferência física para nenhuma orientação. O desvio padrão próximo de 104° (valor esperado para distribuição uniforme em 360°) confirma a uniformidade.

**Distribuição:** Aproximadamente uniforme. **Assimetria:** próxima de 0. **Nota:** variáveis angulares periódicas requerem transformação cosseno/seno antes da modelagem (para evitar descontinuidade em 0°/360°). **Variável de baixa prioridade para modelagem.**

#### **13. Como se distribui a orientação do ponto mais próximo ao Sol dentro da órbita de cada asteroide (`w`)?**

Esperamos distribuição uniforme, similar a `om` — o argumento do periélio também não deve ter preferência física.

In [ ]:
data_w = df_clean['w'].dropna()
fig, ax = plt.subplots(figsize=(9, 4))
sns.histplot(data_w, bins=36, kde=True, ax=ax, color='#4A5F9E', edgecolor='#5A6A7A')
ax.set_title('Argumento do Periélio (w)')
ax.set_xlabel('Argumento do periélio (graus)'); ax.set_ylabel('Contagem')
ax.spines['bottom'].set_visible(False); ax.yaxis.grid(True, linestyle='--', alpha=0.7); ax.set_axisbelow(True)
plt.tight_layout(); plt.show()
print(f"Média: {data_w.mean():.2f}° | Mediana: {data_w.median():.2f}° | Desvio padrão: {data_w.std():.2f}°")
print(f"Assimetria: {skew(data_w):.3f} | Curtose: {kurtosis(data_w):.3f}")

Similar a `om`, a distribuição é aproximadamente uniforme. Mesma consideração sobre transformação cosseno/seno para modelagem.

**Distribuição:** Aproximadamente uniforme. **Variável de baixa prioridade para modelagem.**

#### **14. Em que posição da órbita os asteroides próximos à Terra estavam no momento do catálogo (`ma`)?**

Esperamos distribuição uniforme — os asteroides são observados em posições aleatórias de suas órbitas, sem concentração em nenhum ponto específico.

In [ ]:
data_ma = df_clean['ma'].dropna()
fig, ax = plt.subplots(figsize=(9, 4))
sns.histplot(data_ma, bins=36, kde=True, ax=ax, color='#4A5F9E', edgecolor='#5A6A7A')
ax.set_title('Anomalia Média (ma)')
ax.set_xlabel('Anomalia média (graus)'); ax.set_ylabel('Contagem')
ax.spines['bottom'].set_visible(False); ax.yaxis.grid(True, linestyle='--', alpha=0.7); ax.set_axisbelow(True)
plt.tight_layout(); plt.show()
print(f"Média: {data_ma.mean():.2f}° | Mediana: {data_ma.median():.2f}° | Desvio padrão: {data_ma.std():.2f}°")
print(f"Assimetria: {skew(data_ma):.3f} | Curtose: {kurtosis(data_ma):.3f}")

A anomalia média representa a posição do asteroide na órbita em um momento específico. A distribuição é aproximadamente uniforme — consistente com a expectativa de observações em posições aleatórias.

**Distribuição:** Uniforme. **Nota:** variável temporal de posição, sem poder preditivo para classificação de risco. **Variável de baixa prioridade — provavelmente excluir.**

#### **15. Quanto tempo os asteroides próximos à Terra levam para dar uma volta completa ao redor do Sol (`per_y`)?**

Esperamos períodos concentrados entre 1 e 4 anos, compatíveis com a região interna do sistema solar. Asteroides com períodos próximos a 1 ano merecem atenção especial — orbitam de forma quase sincronizada com a Terra.

In [ ]:
data_per = df_clean['per_y'].dropna()
p99 = data_per.quantile(0.99)
fig, ax = plt.subplots(figsize=(9, 4))
sns.histplot(data_per[data_per <= p99], bins=30, kde=True, ax=ax, color='#4A5F9E', edgecolor='#5A6A7A')
ax.axvline(1.0, color='#E07A5F', linestyle='--', linewidth=1.5, label='1 ano (Terra)')
ax.set_title('Período Orbital (per_y)'); ax.set_xlabel('Período orbital (anos)'); ax.set_ylabel('Contagem')
ax.legend(); ax.spines['bottom'].set_visible(False); ax.yaxis.grid(True, linestyle='--', alpha=0.7); ax.set_axisbelow(True)
plt.tight_layout(); plt.show()
print(f"Média: {data_per.mean():.2f} anos | Mediana: {data_per.median():.2f} anos | Desvio padrão: {data_per.std():.2f} anos")
print(f"Assimetria: {skew(data_per):.3f} | Curtose: {kurtosis(data_per):.3f}")

Distribuição assimétrica à direita, com média ~2,4 anos e mediana ~2,2 anos. Pico próximo a 1,5 anos. O desvio padrão de ~1,1 anos reflete a variação entre subclasses NEO.

**Distribuição:** Assimétrica positiva. **Nota:** `per_y` é matematicamente derivado de `a` (T² ∝ a³) — manter apenas um na modelagem. **Variável parcialmente importante.**

#### **16. Com que velocidade os asteroides próximos à Terra se movem em suas órbitas (`n`)?**

Esperamos distribuição assimétrica — a maioria dos NEOs se move lentamente (períodos longos), com uma minoria de asteroides rápidos de órbitas pequenas.

In [ ]:
data_n = df_clean['n'].dropna()
p99 = data_n.quantile(0.99)
fig, ax = plt.subplots(figsize=(9, 4))
sns.histplot(data_n[data_n <= p99], bins=25, kde=True, ax=ax, color='#4A5F9E', edgecolor='#5A6A7A')
ax.set_title('Movimento Médio Diário (n)'); ax.set_xlabel('Movimento médio diário (graus/dia)'); ax.set_ylabel('Contagem')
ax.spines['bottom'].set_visible(False); ax.yaxis.grid(True, linestyle='--', alpha=0.7); ax.set_axisbelow(True)
plt.tight_layout(); plt.show()
print(f"Média: {data_n.mean():.3f}°/dia | Mediana: {data_n.median():.3f}°/dia | Desvio padrão: {data_n.std():.3f}°/dia")
print(f"Assimetria: {skew(data_n):.3f} | Curtose: {kurtosis(data_n):.3f}")

Distribuição assimétrica à direita, com pico em ~0,3°/dia. Valores mais altos representam asteroides com órbitas menores e mais rápidas (Atens).

**Distribuição:** Assimétrica positiva. **Nota:** `n = 360 / per_y` — diretamente derivado de `per_y` e `a`. Redundante na modelagem. **Variável de baixa prioridade.**

#### **17. As trajetórias dos asteroides próximos à Terra foram calculadas com precisão (`rms`)?**

Esperamos que a maioria tenha soluções orbitais bem determinadas, com uma pequena fração de objetos mal observados gerando outliers.

In [ ]:
data_rms = df_clean['rms'].dropna()
Q1=data_rms.quantile(0.25); Q3=data_rms.quantile(0.75); IQR=Q3-Q1
outliers=data_rms[data_rms > Q3+1.5*IQR]
fig, ax = plt.subplots(figsize=(9, 4))
sns.boxplot(x=data_rms, ax=ax, color='#4A5F9E', flierprops=dict(markerfacecolor='#8FA8D6', markersize=4))
ax.set_title(f'Qualidade da Solução Orbital (rms) — {len(outliers)} outliers ({len(outliers)/len(data_rms)*100:.1f}%)')
ax.set_xlabel('RMS (resíduo médio quadrático, arcsec)'); ax.spines['left'].set_visible(False)
ax.xaxis.grid(True, linestyle='--', alpha=0.7); ax.set_axisbelow(True)
plt.tight_layout(); plt.show()
print(f"Média: {data_rms.mean():.3f} | Mediana: {data_rms.median():.3f} | Desvio padrão: {data_rms.std():.3f}")
print(f"Assimetria: {skew(data_rms):.3f} | Curtose: {kurtosis(data_rms):.3f}")
print(f"Q1: {Q1:.3f} | Q3: {Q3:.3f} | Limite superior Tukey: {Q3+1.5*IQR:.3f}")

O `rms` mede a qualidade do ajuste orbital. A maioria dos objetos concentra-se entre 0,40–0,56 (IQR), com 454 outliers (2,0%) acima de 0,79. O registro extremo (rms > 100) já foi removido na limpeza.

**Distribuição:** Assimétrica positiva. **Problema de qualidade:** 2% dos registros têm soluções menos confiáveis. **Variável de diagnóstico — não usar como feature na modelagem.**

#### **18. Quão confiável é a medição do formato orbital (`sigma_e`) de cada asteroide?**

Esperamos distribuição fortemente assimétrica — a maioria dos objetos tem excentricidade bem determinada, com poucos casos de alta incerteza.

In [ ]:
data_sigma_e = df_clean['sigma_e'].dropna()
p99 = data_sigma_e.quantile(0.99)
Q1=data_sigma_e.quantile(0.25); Q3=data_sigma_e.quantile(0.75); IQR=Q3-Q1
outliers=data_sigma_e[data_sigma_e > Q3+1.5*IQR]
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle('Incerteza da Excentricidade (sigma_e)', fontsize=13, y=1.02)
sns.histplot(data_sigma_e[data_sigma_e <= p99], bins=30, kde=True, ax=axes[0], color='#4A5F9E', edgecolor='#5A6A7A')
axes[0].set_title('Distribuição (até p99)'); axes[0].set_xlabel('sigma_e'); axes[0].set_ylabel('Contagem')
axes[0].spines['bottom'].set_visible(False); axes[0].yaxis.grid(True, linestyle='--', alpha=0.7); axes[0].set_axisbelow(True)
sns.boxplot(x=data_sigma_e, ax=axes[1], color='#4A5F9E', flierprops=dict(markerfacecolor='#8FA8D6', markersize=4))
axes[1].set_title(f'Tukey — {len(outliers)} outliers ({len(outliers)/len(data_sigma_e)*100:.1f}%)'); axes[1].set_xlabel('sigma_e')
axes[1].spines['left'].set_visible(False); axes[1].xaxis.grid(True, linestyle='--', alpha=0.7); axes[1].set_axisbelow(True)
plt.tight_layout(); plt.show()
print(f"Média: {data_sigma_e.mean():.6f} | Mediana: {data_sigma_e.median():.6f} | Desvio padrão: {data_sigma_e.std():.6f}")
print(f"Assimetria: {skew(data_sigma_e):.3f} | Curtose: {kurtosis(data_sigma_e):.3f}")

`sigma_e` representa a incerteza na determinação da excentricidade. Distribuição fortemente assimétrica à direita — maioria dos valores próximos de zero. Outliers representam asteroides com poucos dados observacionais.

**Distribuição:** Exponencial/assimétrica positiva extrema. **Possível transformação:** log. **Variável de diagnóstico — baixa prioridade como feature.**

#### **19. Quando cada asteroide próximo à Terra passou pelo ponto mais próximo do Sol (`tp`)?**

Esperamos uma distribuição que reflete o período de observação do catálogo NASA, sem padrão relevante para classificação de risco.

In [ ]:
data_tp = df_clean['tp'].dropna()
fig, ax = plt.subplots(figsize=(9, 4))
sns.histplot(data_tp, bins=40, kde=True, ax=ax, color='#4A5F9E', edgecolor='#5A6A7A')
ax.set_title('Tempo do Periélio (tp)'); ax.set_xlabel('Tempo do periélio (Dias Julianos)'); ax.set_ylabel('Contagem')
ax.spines['bottom'].set_visible(False); ax.yaxis.grid(True, linestyle='--', alpha=0.7); ax.set_axisbelow(True)
plt.tight_layout(); plt.show()
print(f"Média: {data_tp.mean():.2f} JD | Mediana: {data_tp.median():.2f} JD | Desvio padrão: {data_tp.std():.2f} JD")
print(f"Assimetria: {skew(data_tp):.3f} | Curtose: {kurtosis(data_tp):.3f}")

O tempo do periélio indica quando o asteroide passou pela posição mais próxima do Sol. A distribuição reflete o período de observação do catálogo NASA. **Nota:** variável temporal dependente da época de referência — não representa propriedade intrínseca da órbita.

**Distribuição:** Dependente do histórico de catalogação. **Variável a excluir da modelagem.**

#### **20. Quando foram calculadas as trajetórias dos asteroides próximos à Terra (`epoch`)?**

Esperamos uma distribuição que reflete as campanhas de atualização do catálogo NASA, sem relação com periculosidade.

In [ ]:
data_epoch = df_clean['epoch'].dropna()
fig, ax = plt.subplots(figsize=(9, 4))
sns.histplot(data_epoch, bins=30, kde=True, ax=ax, color='#4A5F9E', edgecolor='#5A6A7A')
ax.set_title('Época da Solução Orbital (epoch)'); ax.set_xlabel('Época (Dias Julianos)'); ax.set_ylabel('Contagem')
ax.spines['bottom'].set_visible(False); ax.yaxis.grid(True, linestyle='--', alpha=0.7); ax.set_axisbelow(True)
plt.tight_layout(); plt.show()
print(f"Média: {data_epoch.mean():.2f} JD | Mediana: {data_epoch.median():.2f} JD | Desvio padrão: {data_epoch.std():.2f} JD")
print(f"Assimetria: {skew(data_epoch):.3f} | Curtose: {kurtosis(data_epoch):.3f}")

A época (`epoch`) é a data de referência para os elementos orbitais. Reflete campanhas de observação e atualizações do catálogo. **Nota:** variável administrativa — não representa propriedade física do asteroide.

**Distribuição:** Dependente do histórico de catalogação. **Variável a excluir da modelagem.**

---

## Considerações Finais da Análise Univariada

### Variáveis de Interesse para Modelagem

Com base nas 20 análises acima, as variáveis são classificadas em três grupos:

**Alta prioridade — features principais:**
- `H` — critério formal de PHA (`H ≤ 22`); distribuição gaussiana; forte discriminador entre classes
- `moid` — critério formal de PHA (`moid ≤ 0,05 AU`); assimétrico; transformação log recomendada
- `e` — forte discriminador visual entre PHAs e não-PHAs confirmado pelo violin plot; assimétrico
- `a` — define o tamanho da órbita; bimodal (reflete subclasses NEO); relacionado a `per_y` e `n`

**Prioridade média — features complementares:**
- `moid_ld` — equivalente a `moid` em distâncias lunares; manter apenas um dos dois na modelagem
- `q` — periélio complementa `a` e `e`; outliers relevantes identificados (Atens/Atiras)
- `i` — inclinação mostra separação parcial entre classes no violin; assimétrica
- `class` — variável categórica com informação sobre tipo orbital; requer encoding

**Baixa prioridade ou excluir:**
- `ad` — derivado de `a` e `e`; colinear; excluir se ambos forem mantidos
- `per_y` e `n` — derivados de `a` pela terceira lei de Kepler; manter no máximo um
- `om`, `w`, `ma` — ângulos de orientação; distribuições uniformes; baixo poder discriminante; requerem transformação sen/cos
- `sigma_e`, `rms` — variáveis de qualidade/diagnóstico, não features preditivas
- `tp`, `epoch` — temporais/administrativas; sem relação com periculosidade intrínseca

### Problemas de Qualidade Identificados
- **Desbalanceamento severo de classes** (~9% PHAs): tratar com SMOTE ou undersampling
- **Distribuições assimétricas**: `moid`, `moid_ld`, `sigma_e`, `n`, `i`, `ad` — candidatas a transformação logarítmica
- **Multicolinearidade**: `a`↔`ad`↔`per_y`↔`n` são matematicamente relacionados; `moid`↔`moid_ld` são equivalentes
- **Outliers removidos na limpeza**: 1 registro `rms > 100`, 2 registros `ad > 100 AU`
- **Outliers mantidos**: 548 em `q` (2,4%), 815 em `i` (3,6%), 454 em `rms` (2,0%) — fisicamente plausíveis

### Distribuições por Tipo
- **Aproximadamente gaussiana:** `H`
- **Assimétrica positiva:** `moid`, `moid_ld`, `e` (nos NEOs), `ad`, `i`, `n`, `sigma_e`, `rms`
- **Bimodal:** `a`, `per_y`
- **Assimétrica negativa:** `q`
- **Aproximadamente uniforme:** `om`, `w`, `ma`
- **Categórica:** `pha`, `neo`, `class`

# Análises Multivariadas

Esta seção investiga as relações entre variáveis e a variável-alvo (`pha`), cruzando os resultados com as hipóteses declaradas no início do projeto. Todas as análises utilizam `df_equal` — a amostra balanceada com 2.066 PHAs e 2.066 não-PHAs — para garantir comparações visuais justas entre as classes. O teste de Mann-Whitney foi utilizado para verificar a significância estatística das diferenças observadas, por ser adequado para distribuições assimétricas e não depender de suposição de normalidade.

### **1. Asteroides com órbitas mais alongadas tendem a ser mais perigosos?**

**Hipótese:** PHAs devem concentrar excentricidades mais altas — para cruzar a região da Terra, a órbita precisa ser suficientemente elíptica.

**Variáveis:** `e` (excentricidade orbital) e `pha` | **Visualização:** KDE por classe — ideal para comparar a forma e posição de duas distribuições contínuas.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
sns.kdeplot(data=df_equal, x='e', hue='pha', common_norm=False,
            palette=PHA_COLORS, ax=ax, fill=True, alpha=0.3)
ax.set_title('Excentricidade Orbital por Classe de Risco (e)')
ax.set_xlabel('Excentricidade (e) — 0 = circular, próximo de 1 = muito elíptica')
ax.set_ylabel('Densidade')
ax.spines['bottom'].set_visible(False)
ax.yaxis.grid(True, linestyle='--', alpha=0.7)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()

**Hipótese confirmada.** O teste de Mann-Whitney retornou p-valor < 0,001, confirmando que a diferença entre as classes é estatisticamente significativa. PHAs têm mediana de excentricidade de 0,557 contra 0,454 dos não-PHAs — órbitas visivelmente mais alongadas. As curvas KDE mostram separação clara: PHAs concentram-se em valores mais altos enquanto não-PHAs têm pico em valores mais baixos. As distribuições se sobrepõem parcialmente, mas a separação é suficiente para tornar `e` um forte discriminador entre as classes.

Isso faz sentido fisicamente: para um asteroide cruzar a região próxima à Terra, sua órbita precisa ser elíptica o suficiente para penetrar na região interna do sistema solar.

---

### **2. Asteroides que orbitam mais próximos ao Sol tendem a ser mais perigosos?**

**Hipótese:** PHAs devem concentrar-se com semi-eixo maior próximo a 1 AU — órbitas menores cruzam mais facilmente a região terrestre.

**Variáveis:** `a` (semi-eixo maior) e `pha` | **Visualização:** KDE por classe.

In [ ]:
data_a = df_equal[df_equal['a'] <= df_equal['a'].quantile(0.99)]

fig, ax = plt.subplots(figsize=(9, 4))
sns.kdeplot(data=data_a, x='a', hue='pha', common_norm=False,
            palette=PHA_COLORS, ax=ax, fill=True, alpha=0.3)
ax.set_title('Semi-eixo Maior por Classe de Risco (a)')
ax.set_xlabel('Semi-eixo maior (AU) — distância média ao Sol')
ax.set_ylabel('Densidade')
ax.spines['bottom'].set_visible(False)
ax.yaxis.grid(True, linestyle='--', alpha=0.7)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()

**Hipótese não confirmada — resultado surpreendente.** O teste de Mann-Whitney retornou p-valor de 0,653, indicando que não há diferença estatisticamente significativa entre PHAs e não-PHAs para o semi-eixo maior. As medianas são praticamente idênticas: 1,731 AU para PHAs e 1,734 AU para não-PHAs.

Isso contraria a intuição inicial de que PHAs deveriam orbitar mais perto do Sol. A explicação está na natureza do subconjunto analisado: após o filtro `neo=Y`, tanto PHAs quanto não-PHAs já pertencem à região próxima à Terra — a diferença de tamanho médio de órbita entre eles desaparece. O que realmente diferencia um PHA de um NEO comum não é o tamanho da órbita, mas sua forma (`e`) e proximidade orbital à Terra (`moid`).

---

### **3. Asteroides que passam mais perto da Terra em sua trajetória orbital são mais perigosos?**

**Hipótese:** PHAs devem ter distância mínima orbital (`moid_ld`) consistentemente menor — é um critério formal de classificação.

**Variáveis:** `moid_ld` (distância mínima à órbita terrestre em distâncias lunares) e `pha` | **Visualização:** boxplot comparativo — adequado para destacar medianas e dispersão entre grupos.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
p99 = df_equal['moid_ld'].quantile(0.99)
data_moid = df_equal[df_equal['moid_ld'] <= p99]

sns.boxplot(data=data_moid, x='moid_ld', y='pha', ax=ax,
            palette=PHA_COLORS,
            flierprops=dict(markerfacecolor='#5A6A7A', markersize=3))
ax.axvline(19.5, color='#E07A5F', linestyle='--', linewidth=1.5, label='Limiar PHA (19,5 LD)')
ax.set_title('Distância Mínima à Órbita Terrestre por Classe de Risco (moid_ld)')
ax.set_xlabel('Distância mínima (distâncias lunares)')
ax.set_ylabel('PHA')
ax.legend()
ax.spines['left'].set_visible(False)
ax.xaxis.grid(True, linestyle='--', alpha=0.7)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()

**Hipótese confirmada — resultado mais forte da análise.** O teste de Mann-Whitney retornou p-valor < 0,001. A mediana de `moid_ld` para PHAs é de 9,1 distâncias lunares, contra 28,0 para não-PHAs — uma diferença de 3x. O boxplot deixa visível que praticamente todos os PHAs estão abaixo do limiar de 19,5 LD, enquanto os não-PHAs se distribuem amplamente acima dele.

`moid_ld` é, de longe, a variável com maior poder de separação entre as classes nesta análise. Isso reforça sua centralidade como critério formal de PHA e como feature essencial para os modelos de classificação.

---

### **4. Asteroides maiores tendem a ser mais perigosos?**

**Hipótese:** PHAs devem ter magnitude absoluta (`H`) menor — objetos maiores e mais brilhantes têm valor de `H` menor, e o critério formal exige `H ≤ 22`.

**Variáveis:** `H` (magnitude absoluta) e `pha` | **Visualização:** KDE por classe com linha do limiar formal.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
sns.kdeplot(data=df_equal, x='H', hue='pha', common_norm=False,
            palette=PHA_COLORS, ax=ax, fill=True, alpha=0.3)
ax.axvline(22, color='#E07A5F', linestyle='--', linewidth=1.5, label='Limiar PHA (H = 22)')
ax.set_title('Magnitude Absoluta por Classe de Risco (H)')
ax.set_xlabel('H (magnitude) — menor valor = objeto maior e mais brilhante')
ax.set_ylabel('Densidade')
ax.legend()
ax.spines['bottom'].set_visible(False)
ax.yaxis.grid(True, linestyle='--', alpha=0.7)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()

**Hipótese confirmada.** O teste de Mann-Whitney retornou p-valor < 0,001. A mediana de `H` para PHAs é 20,3 contra 23,3 para não-PHAs — PHAs são sistematicamente maiores. O KDE mostra separação clara entre as classes, com a distribuição dos PHAs concentrada abaixo do limiar formal de H = 22, enquanto os não-PHAs se deslocam para valores mais altos.

A linha do limiar formal ilustra bem o critério: a grande maioria dos PHAs está à esquerda de H = 22, confirmando que o critério de tamanho é bem calibrado para separar as classes.

---

### **5. Asteroides com órbitas mais inclinadas em relação à Terra tendem a ser menos perigosos?**

**Hipótese:** PHAs devem ter inclinação orbital (`i`) menor — órbitas mais coplanares com a Terra aumentam a probabilidade de cruzamento orbital.

**Variáveis:** `i` (inclinação orbital) e `pha` | **Visualização:** violin por classe — revela a forma completa da distribuição, incluindo assimetrias e multimodalidade.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
sns.violinplot(data=df_equal, x='pha', y='i', ax=ax,
               palette=PHA_COLORS)
ax.set_title('Inclinação Orbital por Classe de Risco (i)')
ax.set_xlabel('PHA')
ax.set_ylabel('Inclinação orbital (graus)')
ax.yaxis.grid(True, linestyle='--', alpha=0.7)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()

**Hipótese parcialmente refutada — resultado não-óbvio.** O teste de Mann-Whitney retornou p-valor de 0,002, confirmando diferença estatisticamente significativa. No entanto, a direção é contrária ao esperado: PHAs têm mediana de inclinação de 9,76°, levemente maior que os 9,30° dos não-PHAs.

A diferença é pequena e ambas as classes concentram-se em inclinações baixas (0°–30°), o que significa que a inclinação por si só não é um bom discriminador. O resultado sugere que, dentro do subconjunto NEO, a inclinação orbital não é o fator determinante para a periculosidade — o que realmente importa é a combinação de excentricidade e proximidade orbital (`moid_ld`).

---

# Visualizações Finais

Esta seção apresenta três visualizações aprimoradas para um público sem conhecimento técnico em astronomia. Cada gráfico é uma versão melhorada de análises já realizadas na seção multivariada, adaptada para comunicar as principais descobertas de forma direta e acessível. Todas utilizam `df_equal` para comparação equilibrada entre as classes.

### **1. O que define um asteroide potencialmente perigoso? Tamanho e proximidade orbital**

Esta visualização combina os dois critérios formais de classificação PHA em um único gráfico: magnitude absoluta (`H`) e distância mínima orbital (`moid_ld`). É uma versão aprimorada da análise multivariada 3 e 4, que investigou cada critério separadamente.

**O que queremos mostrar:** que nem o tamanho isolado nem a proximidade isolada definem o perigo — os dois precisam ocorrer juntos.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

tmp = df_equal[['pha', 'H', 'moid_ld']].dropna()
tmp = tmp[tmp['moid_ld'] > 0].copy()
tmp['log_moid'] = np.log10(tmp['moid_ld'])
ymin = tmp['log_moid'].min() - 0.1
ymax = tmp['log_moid'].max() + 0.1
xmin = tmp['H'].min() - 0.5
xmax = tmp['H'].max() + 0.5

ax.fill_betweenx([ymin, np.log10(19.5)], xmin, 22,
                 color='#ffe0e0', alpha=0.5, zorder=0, label='Zona PHA')

for label, grp in tmp.groupby('pha'):
    ax.scatter(grp['H'], grp['log_moid'],
               c=PHA_COLORS[label], alpha=0.45, s=18, edgecolors='none',
               label='Potencialmente Perigoso' if label == 'Y' else 'Não Perigoso')

ax.axhline(np.log10(19.5), color='#c0392b', linestyle='--', linewidth=1.5, zorder=3)
ax.axvline(22, color='#c0392b', linestyle='--', linewidth=1.5, zorder=3)

ax.text(22.3, ymin + 0.1, 'H = 22\n(≥ 140 m)', color='#c0392b', fontsize=8, va='bottom')
ax.text(xmin + 0.2, np.log10(19.5) + 0.07,
        'MOID = 19,5 distâncias lunares (0,05 AU)', color='#c0392b', fontsize=8)
ax.text(19, np.log10(19.5) - 0.45, 'ZONA PHA',
        color='#c0392b', fontsize=10, fontweight='bold', ha='center')

ax.set_xlim(xmin, xmax)
ax.set_ylim(ymin, ymax)
ax.set_xlabel('Magnitude Absoluta H — quanto menor, maior o asteroide', fontsize=11)
ax.set_ylabel('log₁₀(Distância Mínima à Terra em Distâncias Lunares)\nquanto menor, mais perto da órbita terrestre', fontsize=10)
ax.set_title(
    'O que torna um asteroide potencialmente perigoso?\n'
    'Precisa ser grande o suficiente (H ≤ 22) E passar perto o suficiente da Terra (MOID ≤ 19,5 LD)',
    fontsize=12, fontweight='bold'
)
ax.invert_xaxis()
ax.legend(fontsize=10, framealpha=0.9)
plt.tight_layout()
plt.show()

**Como ler este gráfico:** Cada ponto é um asteroide. Quanto mais à esquerda, maior o objeto. Quanto mais abaixo, mais perto a órbita passa da Terra. A zona rosa marca onde os dois critérios formais de PHA são atendidos ao mesmo tempo.

**Principal descoberta:** PHAs (roxo) concentram-se quase inteiramente dentro da zona rosa — são grandes e passam perto. Não-PHAs (verde) ficam fora dela: ou são pequenos demais, ou orbitam longe o suficiente. Nem o tamanho isolado nem a proximidade isolada definem o perigo — os dois precisam ocorrer juntos.

---

### **2. Como são as órbitas dos asteroides perigosos comparadas às dos demais?**

Esta visualização mostra o formato real das órbitas de 150 asteroides de cada classe ao redor do Sol. É uma versão aprimorada da análise multivariada 1, que identificou a excentricidade como forte discriminador entre as classes.

**O que queremos mostrar:** que PHAs têm órbitas visivelmente mais alongadas que cruzam a região próxima à Terra.

In [ ]:
colors = PHA_COLORS
alphas = {'N': 0.07, 'Y': 0.18}

fig, ax = plt.subplots(figsize=(10, 10))

ax.plot(0, 0, 'o', color='gold', markersize=14, zorder=5)

earth_e = 0.017
earth = MplEllipse(
    xy=(earth_e, 0),
    width=2,
    height=2 * np.sqrt(1 - earth_e**2),
    edgecolor='#6495ED',
    facecolor='none',
    linestyle='--',
    linewidth=1.8
)
ax.add_patch(earth)

for label in ['N', 'Y']:
    sub = ds[ds['pha'] == label].dropna(subset=['a', 'e'])
    sub = sub[(sub['a'] < 6) & (sub['e'] >= 0) & (sub['e'] < 1)]
    sub = sub.sample(n=min(150, len(sub)), random_state=42)

    for _, row in sub.iterrows():
        a_val = row['a']
        e_val = row['e']
        b_val = a_val * np.sqrt(1 - e_val**2)
        c_val = a_val * e_val

        orbit = MplEllipse(
            xy=(c_val, 0),
            width=2 * a_val,
            height=2 * b_val,
            edgecolor=colors[label],
            facecolor='none',
            linewidth=0.9,
            alpha=alphas[label]
        )
        ax.add_patch(orbit)

ax.set_xlim(-3, 7)
ax.set_ylim(-4, 4)
ax.set_aspect('equal')
ax.set_title('Como são as órbitas dos asteroides potencialmente perigosos?\n(Sol no foco, uma curva por asteroide — 150 de cada classe)', fontsize=14)
ax.set_xlabel('Distância horizontal ao Sol (AU)', fontsize=12)
ax.set_ylabel('Distância vertical ao Sol (AU)', fontsize=12)

legend_handles = [
    Line2D([0], [0], color=PHA_COLORS['Y'], lw=2, label='Potencialmente Perigoso (PHA = Y)'),
    Line2D([0], [0], color=PHA_COLORS['N'], lw=2, label='Não Perigoso (PHA = N)'),
    Line2D([0], [0], color='#6495ED', lw=2, ls='--', label='Órbita da Terra (~1 AU)'),
    Line2D([0], [0], marker='o', color='gold', lw=0, ms=10, label='Sol')
]

ax.legend(handles=legend_handles, loc='upper right', fontsize=11)
plt.tight_layout()
plt.show()

**Como ler este gráfico:** O ponto dourado é o Sol. A elipse azul tracejada é a órbita da Terra. Cada curva é a órbita de um asteroide. Órbitas mais circulares ficam próximas ao Sol; órbitas mais alongadas se estendem para longe.

**Principal descoberta:** Asteroides potencialmente perigosos (roxo) têm órbitas visivelmente mais alongadas que cruzam ou tangenciam a órbita da Terra. Os não perigosos (verde) tendem a orbitas mais circulares e distantes, sem cruzar a região terrestre.

---

### **3. Quais asteroides têm órbitas que fisicamente cruzam a distância da Terra?**

Este scatterplot combina o tamanho médio da órbita (`a`) com seu formato (`e`) para identificar visualmente a zona de risco orbital. É uma versão aprimorada da análise multivariada 2, que revelou o resultado surpreendente de que o tamanho da órbita isolado não diferencia PHAs de não-PHAs.

**O que queremos mostrar:** que a combinação de `a` e `e` — e não cada variável isolada — define se uma órbita cruza fisicamente a região da Terra.

In [ ]:
tmp_final = df_equal[['a', 'e', 'pha']].dropna()
tmp_final = tmp_final[(tmp_final['a'] < 5) & (tmp_final['e'] < 0.95)]

fig, ax = plt.subplots(figsize=(12, 7))

for label, grp in tmp_final.groupby('pha'):
    ax.scatter(grp['a'], grp['e'],
               c=PHA_COLORS[label], alpha=0.4, s=20, edgecolors='none',
               label='Potencialmente Perigoso' if label == 'Y' else 'Não Perigoso')

# Terra
ax.scatter(1, 0.017, color='#333333', s=120, zorder=5,
           edgecolors='#333333', linewidth=1, label='Terra (1 AU, e = 0,017)')

# Zona de cruzamento orbital: e > |1 - a| / a
a_range = np.linspace(0.1, 5, 1000)
e_cross = np.abs(1 - a_range) / a_range
e_cross = np.clip(e_cross, 0, 0.95)
ax.fill_between(a_range, e_cross, 0.95, alpha=0.12, color='#c0392b',
                label='Zona de cruzamento orbital')
ax.plot(a_range, e_cross, color='#c0392b', linestyle=':', linewidth=2,
        label='Fronteira: e = |1−a| / a')
ax.axvline(x=1, color='#c0392b', ls='--', alpha=0.4)

ax.set_title('Quais órbitas cruzam fisicamente a região da Terra?\nTamanho médio da órbita vs. formato da órbita', fontsize=13, fontweight='bold')
ax.set_xlabel('Semi-eixo maior (AU) — raio médio da órbita\n1 AU = distância Terra-Sol', fontsize=11)
ax.set_ylabel('Excentricidade — 0 = circular, próximo de 1 = muito alongada', fontsize=11)
ax.set_xlim(0, 5)
ax.set_ylim(0, 0.95)
ax.yaxis.grid(True, linestyle='--', alpha=0.5)
ax.xaxis.grid(True, linestyle='--', alpha=0.5)
ax.set_axisbelow(True)
ax.legend(fontsize=10, loc='upper right')
plt.tight_layout()
plt.show()

**Como ler este gráfico:** Cada ponto é um asteroide. O eixo horizontal é o raio médio da órbita (1 AU = distância Terra-Sol). O eixo vertical é o quão alongada é a órbita. A zona vermelha marca todas as órbitas que fisicamente cruzam a distância da Terra ao Sol. O ponto escuro é a Terra.

**Principal descoberta:** Quase todos os PHAs (roxo) estão dentro da zona vermelha — suas órbitas cruzam fisicamente a região onde a Terra orbita. Os não-PHAs (verde) concentram-se fora dela, com órbitas circulares e distantes. A fronteira de cruzamento separa as duas populações de forma clara. Isso também explica o resultado surpreendente da análise 2: isolado, o tamanho da órbita não separa as classes — é a combinação de tamanho e formato que define o cruzamento orbital.

---

## Síntese e Reflexão Crítica (*Digest*)

```
Este projeto analisou o dataset de asteroides da NASA com o objetivo de compreender quais características físicas e orbitais distinguem asteroides potencialmente perigosos (PHAs) dos demais, como base para a construção de um modelo de classificação supervisionada.

PRINCIPAIS RESULTADOS

A geometria orbital define a periculosidade. As análises multivariadas confirmaram que as variáveis mais discriminantes são a excentricidade (e), o semi-eixo maior (a) e a distância mínima orbital à Terra (moid_ld). A visualização da fronteira de cruzamento orbital (e > |1 − a| / a) demonstrou que quase todos os PHAs ocupam a região de órbitas que cruzam a distância terrestre.

A descoberta mais importante — e não-óbvia — foi que características físicas de superfície (diâmetro e albedo) não distinguem PHAs de não-PHAs. A análise de dispersão diâmetro × albedo mostrou sobreposição completa entre as classes. O critério de periculosidade é definido pela órbita, não pela composição ou tamanho da superfície.

DECISÕES DE ANÁLISE

Verificamos empiricamente que todos os 2.066 PHAs possuem neo=Y — não existe nenhum PHA com neo=N nos dados. Isso fundamentou o uso de neo como variável relevante para análise sem risco de descartar nenhum objeto da classe positiva.

Para comparações visuais entre classes, utilizamos df_sample — uma amostra igualitária criada na limpeza de dados, com todos os PHAs e o mesmo número de não-PHAs sorteados aleatoriamente. Isso garante comparações visuais equilibradas sem distorção por desbalanceamento, sendo usada apenas para visualização.

O DESAFIO DA MODELAGEM

O desbalanceamento extremo de classes (~0,22% PHAs) exigirá técnicas especiais na etapa de Machine Learning: balanceamento com SMOTE ou Near Miss, e métricas adequadas como F1-score, precisão e recall — acurácia isolada é enganosa para problemas tão desbalanceados.

O QUE FARIA DIFERENTE

A análise de correlação de Spearman deveria ter sido realizada antes das visualizações multivariadas individuais, guiando a seleção de variáveis desde o início e evitando análises redundantes entre a, ad, per_y e n. Também subestimamos o impacto da ausência de dados em diameter e albedo — H acabou sendo a única variável física viável como feature.
```

## Machine Learning *(pós-checkpoint)*

Serão desenvolvidos ao menos 3 modelos de classificação para identificar PHAs.

Aspectos a cobrir:
- Seleção de features baseada nas análises exploratórias
- Protocolo de validação adequado para dados desbalanceados
- Técnicas de balanceamento: SMOTE e/ou Near Miss
- Métricas: F1-score, precisão, recall, AUC-ROC
- Comparação com e sem a variável `moid_ld`

In [ ]:
# células de Machine Learning serão adicionadas aqui

## Etapas para Submissão Final

1. Executar todas as células e verificar que os outputs estão visíveis;
2. Salvar como Jupyter Notebook (`.ipynb`);
3. Exportar cópia em PDF (`.pdf`);
4. Compactar notebook, PDF e dataset em `EquipeXX.zip`;
5. Fazer upload no Canvas.